In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install fastapi uvicorn pyngrok nest-asyncio -q

import os, subprocess, threading, time
from pyngrok import ngrok
import warnings
warnings.filterwarnings('ignore')

BASE      = '/content/drive/MyDrive/final_project/'
MODEL_DIR = BASE + 'models/'

# Kill existing tunnels
ngrok.kill()
os.system("pkill -f ngrok")
os.system("pkill -f uvicorn")
time.sleep(3)

# Set token
ngrok.set_auth_token("3CvnZYx9VYDOmbFJIalydxEGEa1_3FxaAw7QZa68rSGbkcDHY")
print("Token set!")

Mounted at /content/drive
Token set!


In [2]:
# ============================================================
# CELL 2 — CREATE app.py
# ============================================================
app_code = '''
import re
import numpy as np
import joblib
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from scipy.sparse import hstack, csr_matrix

# ── Load models ──────────────────────────────────────────────
BASE      = "/content/drive/MyDrive/final_project/"
MODEL_DIR = BASE + "models/"

tfidf        = joblib.load(MODEL_DIR + "tfidf_vectorizer.pkl")
xgb_type     = joblib.load(MODEL_DIR + "xgb_type.pkl")
xgb_priority = joblib.load(MODEL_DIR + "xgb_priority.pkl")
xgb_reg      = joblib.load(MODEL_DIR + "xgb_regressor.pkl")
le_type      = joblib.load(MODEL_DIR + "le_type.pkl")
le_priority  = joblib.load(MODEL_DIR + "le_priority.pkl")
scaler       = joblib.load(MODEL_DIR + "scaler_clf.pkl")

print("All models loaded!")

# ── App ──────────────────────────────────────────────────────
app = FastAPI(
    title       = "Customer Support Intelligence API",
    description = "Predicts ticket type, priority, and resolution time",
    version     = "1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# ── Schemas ───────────────────────────────────────────────────
class TicketRequest(BaseModel):
    ticket_subject     : str
    ticket_description : str
    customer_age       : Optional[float] = 35.0
    ticket_channel     : Optional[str]   = "Email"
    customer_gender    : Optional[str]   = "Other"
    product_purchased  : Optional[str]   = "Unknown"

class PredictionResponse(BaseModel):
    ticket_type            : str
    ticket_type_confidence : float
    ticket_priority        : str
    priority_confidence    : float
    estimated_resolution   : float
    resolution_unit        : str

# ── Keyword signals ───────────────────────────────────────────
KEYWORD_SIGNALS = {
    "billing"      : ["bill","payment","charge","invoice","price",
                      "cost","fee","refund","money","amount","pay"],
    "technical"    : ["error","bug","crash","install","update",
                      "software","hardware","device","broken","fix",
                      "issue","problem","work","connect","battery"],
    "cancellation" : ["cancel","cancelation","subscription","stop",
                      "end","terminate","close","discontinue","quit"],
    "refund"       : ["refund","return","money back","reimburse",
                      "exchange","replace","credit","compensate"],
    "product"      : ["product","feature","how","work","use",
                      "setup","configure","compatible","spec","model"]
}

CHANNEL_MAP = {"Email":0,"Chat":1,"Phone":2,"Social media":3}
GENDER_MAP  = {"Female":0,"Male":1,"Other":2}

# ── Preprocessing ─────────────────────────────────────────────
def preprocess(req: TicketRequest):
    text = (req.ticket_subject + " " +
            req.ticket_description).lower()
    text = re.sub(r"<.*?>",    " ", text)
    text = re.sub(r"http\\S+", " ", text)
    text = re.sub(r"[^a-z\\s]"," ", text)
    text = re.sub(r"\\s+",     " ", text).strip()

    tfidf_vec  = tfidf.transform([text])
    word_count = len(text.split())
    char_count = len(text)
    kw         = [sum(1 for kw in kws if kw in text)
                  for kws in KEYWORD_SIGNALS.values()]

    tabular = np.array([[
        req.customer_age,
        CHANNEL_MAP.get(req.ticket_channel, 0),
        GENDER_MAP.get(req.customer_gender, 2),
        0,
        365,
        word_count,
        char_count,
        0.0
    ]])

    tabular_scaled = scaler.transform(tabular)
    return hstack([tfidf_vec, csr_matrix(tabular_scaled)])

# ── Endpoints ─────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "name"     : "Customer Support Intelligence API",
        "version"  : "1.0.0",
        "endpoints": ["/health", "/predict", "/docs"]
    }

@app.get("/health")
def health():
    return {
        "status" : "healthy",
        "message": "API is running!"
    }

@app.post("/predict", response_model=PredictionResponse)
def predict(req: TicketRequest):
    features = preprocess(req)

    type_probs      = xgb_type.predict_proba(features)[0]
    type_idx        = type_probs.argmax()
    ticket_type     = le_type.classes_[type_idx]
    type_conf       = round(float(type_probs[type_idx]), 4)

    priority_probs  = xgb_priority.predict_proba(features)[0]
    priority_idx    = priority_probs.argmax()
    ticket_priority = le_priority.classes_[priority_idx]
    priority_conf   = round(float(priority_probs[priority_idx]), 4)

    resolution      = round(float(
        xgb_reg.predict(features)[0]), 1)

    return PredictionResponse(
        ticket_type            = ticket_type,
        ticket_type_confidence = type_conf,
        ticket_priority        = ticket_priority,
        priority_confidence    = priority_conf,
        estimated_resolution   = resolution,
        resolution_unit        = "hours"
    )
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print("app.py created!")

app.py created!


In [3]:
# ============================================================
# STEP 1 — check what models exist in your Drive
# ============================================================
import os

MODEL_DIR = '/content/drive/MyDrive/final_project/models/'

print("Files in models folder:")
for f in sorted(os.listdir(MODEL_DIR)):
    print(f"  {f}")

Files in models folder:
  cluster_scaler.pkl
  distilbert
  dt_priority.pkl
  dt_type.pkl
  kmeans_model.pkl
  le_priority.pkl
  le_type.pkl
  lgbm_priority.pkl
  lgbm_regressor.pkl
  lgbm_type.pkl
  lr_priority.pkl
  lr_type.pkl
  minmax_scaler.pkl
  nb_priority.pkl
  nb_type.pkl
  rf_priority.pkl
  rf_type.pkl
  scaler_clf.pkl
  shap_explainer.pkl
  standard_scaler.pkl
  tfidf_vectorizer.pkl
  xgb_priority.pkl
  xgb_regressor.pkl
  xgb_shap.pkl
  xgb_type.pkl


In [4]:
# ============================================================
# STEP 2 — create corrected app.py
# ============================================================

# First check exact BASE path
import os
BASE = '/content/drive/MyDrive/final_project/'
print("BASE exists:", os.path.exists(BASE))
print("Models folder:", os.path.exists(BASE + 'models/'))

# List available scalers
scalers = [f for f in os.listdir(BASE + 'models/')
           if 'scaler' in f.lower()]
print("Scalers found:", scalers)

# List available xgb models
xgb_models = [f for f in os.listdir(BASE + 'models/')
              if 'xgb' in f.lower()]
print("XGB models found:", xgb_models)

BASE exists: True
Models folder: True
Scalers found: ['standard_scaler.pkl', 'scaler_clf.pkl', 'minmax_scaler.pkl', 'cluster_scaler.pkl']
XGB models found: ['xgb_regressor.pkl', 'xgb_priority.pkl', 'xgb_type.pkl', 'xgb_shap.pkl']


In [5]:
app_code = '''
import re
import numpy as np
import joblib
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from scipy.sparse import hstack, csr_matrix

# ── Load models ──────────────────────────────────────────────
MODEL_DIR = "/tmp/models/"

tfidf        = joblib.load(MODEL_DIR + "tfidf_vectorizer.pkl")
xgb_type     = joblib.load(MODEL_DIR + "xgb_type.pkl")
xgb_priority = joblib.load(MODEL_DIR + "xgb_priority.pkl")
xgb_reg      = joblib.load(MODEL_DIR + "xgb_regressor.pkl")
le_type      = joblib.load(MODEL_DIR + "le_type.pkl")
le_priority  = joblib.load(MODEL_DIR + "le_priority.pkl")
scaler       = joblib.load(MODEL_DIR + "standard_scaler.pkl")

print("All models loaded!")

# ── App ──────────────────────────────────────────────────────
app = FastAPI(
    title       = "Customer Support Intelligence API",
    description = "Predicts ticket type, priority and resolution time",
    version     = "1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Schemas ───────────────────────────────────────────────────
class TicketRequest(BaseModel):
    ticket_subject     : str
    ticket_description : str
    customer_age       : Optional[float] = 35.0
    ticket_channel     : Optional[str]   = "Email"
    customer_gender    : Optional[str]   = "Other"
    product_purchased  : Optional[str]   = "Unknown"

class PredictionResponse(BaseModel):
    ticket_type            : str
    ticket_type_confidence : float
    ticket_priority        : str
    priority_confidence    : float
    estimated_resolution   : float
    resolution_unit        : str

# ── Keyword signals ───────────────────────────────────────────
KEYWORD_SIGNALS = {
    "billing"      : ["bill","payment","charge","invoice","price",
                      "cost","fee","refund","money","amount","pay"],
    "technical"    : ["error","bug","crash","install","update",
                      "software","hardware","device","broken","fix",
                      "issue","problem","work","connect","battery"],
    "cancellation" : ["cancel","cancelation","subscription","stop",
                      "end","terminate","close","discontinue","quit"],
    "refund"       : ["refund","return","money back","reimburse",
                      "exchange","replace","credit","compensate"],
    "product"      : ["product","feature","how","work","use",
                      "setup","configure","compatible","spec","model"]
}

CHANNEL_MAP = {"Email":0,"Chat":1,"Phone":2,"Social media":3}
GENDER_MAP  = {"Female":0,"Male":1,"Other":2}

# ── Check scaler feature count ────────────────────────────────
SCALER_FEATURES = scaler.n_features_in_
print(f"Scaler expects {SCALER_FEATURES} features")

# ── Preprocessing ─────────────────────────────────────────────
def preprocess(req: TicketRequest):
    # Clean text
    text = (req.ticket_subject + " " +
            req.ticket_description).lower()
    text = re.sub(r"<.*?>",     " ", text)
    text = re.sub(r"http\\S+",  " ", text)
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"\\s+",      " ", text).strip()

    # TF-IDF
    tfidf_vec  = tfidf.transform([text])

    # Numeric features
    word_count = len(text.split())
    char_count = len(text)

    # Base tabular (8 features)
    tabular_8 = [
        req.customer_age,
        CHANNEL_MAP.get(req.ticket_channel, 0),
        GENDER_MAP.get(req.customer_gender, 2),
        0,        # product_encoded
        365,      # ticket_age_days
        word_count,
        char_count,
        0.0       # sentiment_score
    ]

    # Add keyword features if scaler expects more than 8
    if SCALER_FEATURES > 8:
        kw = [sum(1 for kw in kws if kw in text)
              for kws in KEYWORD_SIGNALS.values()]
        tabular_full = tabular_8 + kw
    else:
        tabular_full = tabular_8

    tabular        = np.array([tabular_full[:SCALER_FEATURES]])
    tabular_scaled = scaler.transform(tabular)

    return hstack([tfidf_vec, csr_matrix(tabular_scaled)])

# ── Endpoints ─────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "name"     : "Customer Support Intelligence API",
        "version"  : "1.0.0",
        "endpoints": ["/health", "/predict", "/docs"]
    }

@app.get("/health")
def health():
    return {
        "status" : "healthy",
        "message": "API is running!"
    }

@app.post("/predict", response_model=PredictionResponse)
def predict(req: TicketRequest):
    features = preprocess(req)

    # Ticket type
    type_probs      = xgb_type.predict_proba(features)[0]
    type_idx        = type_probs.argmax()
    ticket_type     = le_type.classes_[type_idx]
    type_conf       = round(float(type_probs[type_idx]), 4)

    # Ticket priority
    priority_probs  = xgb_priority.predict_proba(features)[0]
    priority_idx    = priority_probs.argmax()
    ticket_priority = le_priority.classes_[priority_idx]
    priority_conf   = round(float(priority_probs[priority_idx]), 4)

    # Resolution time
    resolution      = round(float(
        xgb_reg.predict(features)[0]), 1)

    return PredictionResponse(
        ticket_type            = ticket_type,
        ticket_type_confidence = type_conf,
        ticket_priority        = ticket_priority,
        priority_confidence    = priority_conf,
        estimated_resolution   = resolution,
        resolution_unit        = "hours"
    )
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print("app.py created with correct model paths!")
print("Models used:")
print("  tfidf_vectorizer.pkl")
print("  xgb_type.pkl")
print("  xgb_priority.pkl")
print("  xgb_regressor.pkl")
print("  standard_scaler.pkl")
print("  le_type.pkl")
print("  le_priority.pkl")

app.py created with correct model paths!
Models used:
  tfidf_vectorizer.pkl
  xgb_type.pkl
  xgb_priority.pkl
  xgb_regressor.pkl
  standard_scaler.pkl
  le_type.pkl
  le_priority.pkl


In [6]:
# ============================================================
# CELL 4 — LAUNCH API
# ============================================================
import os, subprocess, threading, time
from pyngrok import ngrok

# Kill existing
ngrok.kill()
os.system("pkill -f ngrok")
os.system("pkill -f uvicorn")
time.sleep(3)

# Create local model directory and copy models from Drive
LOCAL_MODEL_DIR = "/tmp/models/"
if not os.path.exists(LOCAL_MODEL_DIR):
    os.makedirs(LOCAL_MODEL_DIR)

SOURCE_MODEL_DIR = "/content/drive/MyDrive/final_project/models/"
models_to_copy = [
    "tfidf_vectorizer.pkl",
    "xgb_type.pkl",
    "xgb_priority.pkl",
    "xgb_regressor.pkl",
    "le_type.pkl",
    "le_priority.pkl",
    "standard_scaler.pkl"
]

for model_file in models_to_copy:
    src_path = os.path.join(SOURCE_MODEL_DIR, model_file)
    dst_path = os.path.join(LOCAL_MODEL_DIR, model_file)
    if os.path.exists(src_path):
        os.system(f"cp {src_path} {dst_path}")
    else:
        print(f"Warning: Model file not found at {src_path}")
time.sleep(1) # Give a moment for copying to complete


# Set token
NGROK_TOKEN = "3CvnZYx9VYDOmbFJIalydxEGEa1_3FxaAw7QZa68rSGbkcDHY"
ngrok.set_auth_token(NGROK_TOKEN)

# Launch uvicorn
process = subprocess.Popen(
    ["python", "-m", "uvicorn",
     "app:app",
     "--host", "0.0.0.0",
     "--port", "8000",
     "--loop", "asyncio"],     # ← fixes the asyncio conflict
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd="/content"
)
time.sleep(15) # Increased sleep time to ensure uvicorn starts fully

# Check if running
if process.poll() is None:
    print("uvicorn started!")
else:
    out, err = process.communicate()
    print("STDOUT:", out.decode())
    print("STDERR:", err.decode())

# Create tunnel
public_url = ngrok.connect(8000)
API_URL    = str(public_url).replace(
    'NgrokTunnel: "','').split('"')[0]

print(f"\n{'='*55}")
print(f"PUBLIC API URL : {API_URL}")
print(f"Docs URL       : {API_URL}/docs")
print(f"Health check   : {API_URL}/health")
print(f"{'='*55}")

uvicorn started!

PUBLIC API URL : https://unnatural-whole-geometry.ngrok-free.dev
Docs URL       : https://unnatural-whole-geometry.ngrok-free.dev/docs
Health check   : https://unnatural-whole-geometry.ngrok-free.dev/health


In [7]:
# ============================================================
# CELL 5 — TEST API
# ============================================================
import requests, time
time.sleep(3)

headers = {"ngrok-skip-browser-warning": "true"}

# Health check
print("Test 1 — Health check:")
r = requests.get(f"{API_URL}/health",
                 headers=headers, timeout=10)
print(r.json())

# Billing ticket
print("\nTest 2 — Billing ticket:")
r = requests.post(f"{API_URL}/predict",
    headers=headers, timeout=10,
    json={
        "ticket_subject"     : "Problem with my invoice",
        "ticket_description" : "I was charged twice for my subscription this month. I need a refund.",
        "customer_age"       : 34,
        "ticket_channel"     : "Email",
        "customer_gender"    : "Female"
    })
print(r.json())

# Technical ticket
print("\nTest 3 — Technical ticket:")
r = requests.post(f"{API_URL}/predict",
    headers=headers, timeout=10,
    json={
        "ticket_subject"     : "App keeps crashing",
        "ticket_description" : "Software crashes on install. Error code 404. Device unresponsive.",
        "customer_age"       : 28,
        "ticket_channel"     : "Chat",
        "customer_gender"    : "Male"
    })
print(r.json())

# Cancellation ticket
print("\nTest 4 — Cancellation ticket:")
r = requests.post(f"{API_URL}/predict",
    headers=headers, timeout=10,
    json={
        "ticket_subject"     : "Cancel my subscription",
        "ticket_description" : "I want to cancel and terminate my account immediately.",
        "customer_age"       : 45,
        "ticket_channel"     : "Phone"
    })
print(r.json())

# Save URL
with open('/content/drive/MyDrive/final_project/api_url.txt', 'w') as f:
    f.write(API_URL)

print(f"\nAPI URL saved: {API_URL}")
print(f"Swagger docs : {API_URL}/docs")
print("\nNext → MLflow + Dashboard!")

Test 1 — Health check:
{'status': 'healthy', 'message': 'API is running!'}

Test 2 — Billing ticket:
{'ticket_type': 'Cancellation request', 'ticket_type_confidence': 0.3419, 'ticket_priority': 'Medium', 'priority_confidence': 0.4072, 'estimated_resolution': 45.1, 'resolution_unit': 'hours'}

Test 3 — Technical ticket:
{'ticket_type': 'Cancellation request', 'ticket_type_confidence': 0.2394, 'ticket_priority': 'High', 'priority_confidence': 0.4459, 'estimated_resolution': 41.1, 'resolution_unit': 'hours'}

Test 4 — Cancellation ticket:
{'ticket_type': 'Cancellation request', 'ticket_type_confidence': 0.2922, 'ticket_priority': 'Low', 'priority_confidence': 0.3457, 'estimated_resolution': 55.2, 'resolution_unit': 'hours'}

API URL saved: https://unnatural-whole-geometry.ngrok-free.dev
Swagger docs : https://unnatural-whole-geometry.ngrok-free.dev/docs

Next → MLflow + Dashboard!
